# SITELLE SN3-Like Benchmark

This notebook builds a conventional filtered imaging-FTS baseline using an SN3-like bandpass and scan geometry.

Because the public SITELLE ETC is browser-driven, this notebook prepares a simulator-side reference case and prints the quantities needed for a manual cross-check against the external ETC.

In [ ]:
import numpy as np

from mkid_ifts_sim import InstrumentConfig, load_template, snr_from_time

# SN3-like setup from the project plan: 648-685 nm bandpass and delta_x=2943 nm.
# The simulator emulates the bandpass by restricting the spectral grid to the SN3 window.
sn3_config = InstrumentConfig(
    sigma_min_cm=1.0e7 / 685.0,
    sigma_max_cm=1.0e7 / 648.0,
    delta_x_m=2.943e-6,
    n_steps=350,
    t_exp_per_step_s=5.0,
    dual_output=False,
    strategy="probabilistic",
    R_energy_ref=1.0e6,
    apodization="none",
)
source = load_template("stellar_g2v", sigma_grid_cm=sn3_config.sigma_grid(), magnitude=20.0, band="r")
result = snr_from_time(source, sn3_config, sn3_config.total_observing_time_s)

In [ ]:
reference_wavelengths = [650.0, 656.3, 672.0]
for ref_nm in reference_wavelengths:
    snr = np.interp(ref_nm, result.wavelength_nm[::-1], result.snr[::-1])
    print(f"SN3-like benchmark SNR at {ref_nm:.1f} nm: {snr:.3f}")

print(f"Total observing time: {sn3_config.total_observing_time_s:.1f} s")
print(f"Free spectral range: {sn3_config.free_spectral_range_cm:.1f} cm^-1")
print(f"Spectral window: {648.0:.1f}-{685.0:.1f} nm")

## Manual Comparison Checklist

1. Open the public SITELLE ETC and select `SN3 (648-685 nm)`.
2. Enter a G2V star with `r = 20.0` and `5 s` exposure per step.
3. Set the reference wavelength to one of the values printed above.
4. Compare the ETC SNR to the simulator output from this notebook.

The aim of this benchmark is not perfect equality; it is to keep the simulator anchored to a conventional filtered imaging-FTS baseline.